## **Subset selection**

In [ ]:
import pandas as pd

INPUT_PATH = "/content/Amazon_CM_37K.csv"
OUTPUT_PATH = "Amazon_CM_10K.csv"

TEXT_COL = "malayalam_reviews"

TARGET_SIZE = 10000
MAX_LEN = 60
MIN_LEN = 3

def tokenize(sentence):
    return str(sentence).split()

def get_length(sentence):
    return len(tokenize(sentence))

# 3-10, 11-25, 26-60
def length_category(length):
    if length <= 10:
        return "short"
    elif length <= 25:
        return "medium"
    else:
        return "long"

df = pd.read_csv(INPUT_PATH)
df["length"] = df[TEXT_COL].apply(get_length)

df = df[
    (df["length"] >= MIN_LEN) &
    (df["length"] <= MAX_LEN)
].copy()

df["length_category"] = df["length"].apply(length_category)

counts = df["length_category"].value_counts()
proportions = counts / len(df)

target_counts = (proportions * TARGET_SIZE).round().astype(int)

diff = TARGET_SIZE - target_counts.sum()
subset_list = []

for category, n_samples in target_counts.items():
    category_df = df[df["length_category"] == category]
    subset_list.append(category_df.sample(n=n_samples, random_state=42))

subset = pd.concat(subset_list).sample(frac=1, random_state=42).reset_index(drop=True)
subset.to_csv(OUTPUT_PATH, index=True)

# compare original vs subset distribution
orig_dist = df["length_category"].value_counts(normalize=True)
subset_dist = subset["length_category"].value_counts(normalize=True)

comparison = pd.DataFrame({
    "original": orig_dist,
    "subset": subset_dist
}).fillna(0)

print(comparison)

## **Train-test split**

In [ ]:
import pandas as pd
import torch

INPUT_PATH = "Amazon_CM_10K.csv"

df = pd.read_csv(INPUT_PATH)

torch.manual_seed(42)

N = len(df)
test_size = int(0.1 * N)

indices = torch.randperm(N)

test_indices = indices[:test_size].tolist()
train_indices = indices[test_size:].tolist()

train_df = df.iloc[train_indices].reset_index(drop=True)
test_df = df.iloc[test_indices].reset_index(drop=True)

train_df.to_csv("Amazon_CM_Train.csv", index=False)
test_df.to_csv("Amazon_CM_Test.csv", index=False)

# distribution
train_dist = train_df["length_category"].value_counts(normalize=True)
test_dist = test_df["length_category"].value_counts(normalize=True)

comparison = pd.DataFrame({
    "train": train_dist,
    "test": test_dist
}).fillna(0)

print("Train size:", len(train_df))
print("Test size:", len(test_df))
print(comparison)

In [ ]:
import pandas as pd

df = pd.read_csv("Amazon_CM_Test.csv")
df['target'] = df["english_reviews"]
df['source'] = df["malayalam_reviews"]
df = df[['source','target']]
df.to_csv("test.csv")

In [ ]:
# bidirectional splitting and reversed merging
import pandas as pd
import torch

INPUT_PATH = "Amazon_CM_Train.csv"   # your 9K train

df = pd.read_csv(INPUT_PATH)

torch.manual_seed(42)

N = len(df)
subset_size = int(0.5 * N)   # 4.5K from 9K

indices = torch.randperm(N)

subset_indices = indices[:subset_size].tolist()
remaining_indices = indices[subset_size:].tolist()

subset_df = df.iloc[subset_indices].reset_index(drop=True)
remaining_df = df.iloc[remaining_indices].reset_index(drop=True)

# save
subset_df.to_csv("Amazon_CM_Train_4_5K.csv", index=False)
remaining_df.to_csv("Amazon_CM_Remaining_4_5K.csv", index=False)

# distribution check
subset_dist = subset_df["length_category"].value_counts(normalize=True)
remaining_dist = remaining_df["length_category"].value_counts(normalize=True)

comparison = pd.DataFrame({
    "subset_4.5k": subset_dist,
    "remaining_4.5k": remaining_dist
}).fillna(0)

print("Subset size:", len(subset_df))
print("Remaining size:", len(remaining_df))
print(comparison)

In [ ]:
import pandas as pd

INPUT_PATH = "Amazon_CM_Train_4_5K.csv"
OUTPUT_PATH = "Amazon_CM_Bidirectional_9K.csv"

df = pd.read_csv(INPUT_PATH)

data = []

for _, row in df.iterrows():
    manglish = str(row["malayalam_reviews"]).strip()
    english = str(row["english_reviews"]).strip()

    # Manglish → English
    data.append({
        "source": "<M2E> " + manglish,
        "target": english
    })

    # English → Manglish
    data.append({
        "source": "<E2M> " + english,
        "target": manglish
    })

bidirectional_df = pd.DataFrame(data)

# shuffle for better training
bidirectional_df = bidirectional_df.sample(frac=1, random_state=42).reset_index(drop=True)

# save
bidirectional_df.to_csv(OUTPUT_PATH, index=False)

print("Final dataset size:", len(bidirectional_df))
print(bidirectional_df.head())

# **LLMs**

In [ ]:
!pip install groq pandas

In [ ]:
GROQ = "API KEY"

from groq import Groq

client = Groq(api_key= GROQ)

def translate_groq(text, model):
    prompt = f"""
    You are a strict translation system.

    Task: Translate the given sentence into natural, grammatically correct English.

    Rules:
    - Preserve the original meaning exactly
    - Do NOT add or remove information
    - Do NOT paraphrase beyond necessary grammar correction
    - Fix grammar, spelling, and fluency in English
    - Use proper sentence casing and punctuation
    - Keep tone neutral and natural

    Output:
    - Only the final corrected English sentence
    - No explanation
    - No extra text

    Examples:

    1.Input: നന്നായി നിർമ്മിച്ചതും ഓരോ ചില്ലിക്കാശിനും വിലയുള്ളതുമായ മികച്ച ടാബ്‌ലെറ്റ്
    Output: Great tablet for the price well built and worth every penny

    2.Input: ഇതിന് രക്ഷാകർതൃ നിയന്ത്രണവും കർഫ്യൂവും ഉണ്ടെന്ന് ഞാൻ ഇഷ്ടപ്പെടുന്നു, എനിക്ക് ഇഷ്ടപ്പെടാത്തത് ചാർജർ അതിൽ കുറവാണെന്ന് തോന്നുന്നു അല്ലെങ്കിൽ വേഗത്തിൽ ചാർജ് ചെയ്യുന്നില്ല, ഇത് സ്ലോ ചാർജ് ആണ്, പക്ഷേ അവൾ അത് ഇഷ്ടപ്പെടുന്നു, അത് കൂടുതൽ ചെയ്യുന്നു ഞാന് പ്രതീക്ഷിച്ചത്. ചൈൽഡ് മോഡ് കാരണം അവൾക്ക് ഇന്റർനെറ്റ് ഉപയോഗിക്കാൻ കഴിയാത്ത പുസ്തകങ്ങൾ ഡൗൺലോഡ് ചെയ്യാൻ കഴിയും, ഇത് കുട്ടികൾക്കും മുതിർന്നവർക്കും ഒരു മികച്ച ടാബ്‌ലെറ്റാണ്
    Output: I like that it has parental control and a curfew on it, what I don't like is the charger it seemed to have a short in it or doesn't charge fast it's a slow charge but she loves it and it does way more then I expected. She is able to download books she can't use the internet because of the child mode it's a great tablet for children and adults

    3.Input: മികച്ച ഓൺലൈൻ അനുഭവം. പ്രായമായവർക്ക് ഇത് എളുപ്പമാക്കിയതിന് നന്ദി, അതിലൂടെ അവർക്ക് എന്തുചെയ്യണമെന്ന് മനസ്സിലാക്കാനാകും. ആകർഷണീയമായ വാങ്ങൽ, വീട്ടിൽ ഷോപ്പുചെയ്യുന്നത് ആകർഷണീയമായിരുന്നു. നന്ദി, ബെസ്റ്റ് ബൈ. രാജ്ഞി
    Output: Great online experience. Thanks for making it easy for older people so they can understand what to do. Awesome buy and it was awesome to shop at home. THANKS, Best Buy. Queenie

    4.Input: ക്രിസ്മസിന് എന്റെ സുഹൃത്ത് എനിക്ക് ഇത് കൊണ്ടുവന്നു. സംഗീതം സ്ട്രീം ചെയ്യുന്നത് വളരെ നല്ലതാണ്. എന്റെ ഭാര്യക്ക് എനിക്ക് ഒരു പ്രൈം അക്കൗണ്ട് ലഭിച്ചു, ഇപ്പോൾ നമുക്ക് സിനിമകൾ കാണാനും സംഗീതം കേൾക്കാനും അലക്‌സയോട് ആവശ്യപ്പെടാനും കഴിയും.
    Output: My buddy got me this for Christmas. It is great to stream music. My wife got me a Prime account and we can now watch movies, listen to music, etc just by asking Alexa.

    5.Input: അത് എത്ര ചെറുതാണെന്നും സ്‌ക്രീൻ നിറത്തിലുള്ളതാണെന്നും എനിക്ക് ശരിക്കും ഇഷ്ടമാണ്.
    Output: I really like how small it is and the fact that the screen is in color.

    6.Input: ബ്ലാക്ക് ഫ്രൈഡേയ്‌ക്ക് ഉൽപ്പന്നം വിൽപ്പനയ്‌ക്കെത്തി. അതിന്റെ ഉദ്ദേശ്യം നിറവേറ്റുന്നു. ഗുണനിലവാരം മികച്ചതാണ്, ഞാൻ ഇത് എന്റെ Chromecast-ൽ ഉപയോഗിക്കുന്നു.
    Output: The product was on sale for Black Friday. Serves its purpose. Quality is good and I use it with my Chromecast.

    7.Input: ക്രിസ്മസിനായി ഞങ്ങൾ ഇത് ഞങ്ങളുടെ മകന് വാങ്ങി, അവന് 9 വയസ്സ്. അവന് ആവശ്യമായത് ഇതാണ്, ധാരാളം ഗെയിമുകൾ, അവന് വീഡിയോകൾ കാണാനാകും, മുതലായവ, പക്ഷേ എളുപ്പത്തിൽ സജ്ജീകരിച്ച രക്ഷാകർതൃ നിയന്ത്രണങ്ങൾക്ക് നന്ദി, ഇത് ഉപയോഗിച്ച് എനിക്ക് അവനിൽ സുരക്ഷിതത്വം തോന്നുന്നു.
    Output: We bought this for our son for Christmas, he's 9. It is exactly what he needed, plenty of games, he can watch videos, etc but I can feel safe with him using it thanks to the easily set parental controls.

    8.Input: ഞാൻ ഇത് എന്റെ ഭാര്യക്ക് ഒരു എക്കോ ഡോട്ടിനൊപ്പം ഒരു സമ്മാനമായി വാങ്ങി. അവൾ അത് ഇഷ്ടപ്പെടുന്നു. സജ്ജീകരിക്കാനും ഉപയോഗിക്കാനും എളുപ്പമാണ്.
    Output: I bought this as a gift for my wife along with an Echo DOT. She loves it. Easy to setup and use.

    9.Input: ക്രിസ്മസിന് കുട്ടികൾക്കായി ഇത് ലഭിച്ചു...ഏറ്റവും മികച്ച സമ്മാനം.
    Output: Got this for the kids for Xmas...was the BEST gift.

    10.Input: എന്റെ പുത്രന്മാരുടെ പ്രൊഫൈലിനായി ഉപകരണം സജ്ജീകരിക്കുന്നതിന് ഞാൻ പ്രതീക്ഷിച്ചതിലും കുറച്ച് സമയമെടുത്തു, പക്ഷേ എനിക്കത് ഇഷ്ടമാണ്. നാവിഗേറ്റ് ചെയ്യാൻ എളുപ്പമാണ്, കൂടാതെ ഒരു ടൺ പ്രീലോഡ് ചെയ്ത പ്രവർത്തനങ്ങളും അവനുവേണ്ടിയുണ്ട്.
    Output: Setting up the device for my sons profile took a litthe more time than I expected, but I love it. It's easy to navigate and has a ton of preloaded activities for him.

    Now translate: "{text}"
    """

    response = client.chat.completions.create(
        model= model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content

In [ ]:
import pandas as pd
import os
import time

INPUT_FILE = "test.csv"
OUTPUT_FILE = "Output_Groq_llama_507_fewshot.csv"
OUTPUT_COL = "llama" #change

SAVE_EVERY = 1
SLEEP_TIME = 0

def load_df():
    input_df = pd.read_csv(INPUT_FILE)

    if os.path.exists(OUTPUT_FILE):
        df = pd.read_csv(OUTPUT_FILE)
        if len(df) < len(input_df):
            missing = input_df.iloc[len(df):].copy()
            missing[OUTPUT_COL] = ""
            df = pd.concat([df, missing], ignore_index=True)
    else:
        df = input_df.copy()
        df[OUTPUT_COL] = ""

    df[OUTPUT_COL] = df.get(OUTPUT_COL, "").astype("object")
    return df


def process(df, model):
    for i, row in df.iterrows():

        if pd.notna(row[OUTPUT_COL]) and str(row[OUTPUT_COL]).strip() != "":
            continue

        try:
            text = row["source"]
            translated = translate_groq(text, model)
            translated = str(translated).strip().replace('"', '')
            df.at[i, OUTPUT_COL] = translated
            print(i, translated)

        except Exception as e:
            print(i, "ERROR:", e)
            continue

        if i % SAVE_EVERY == 0:
            df.to_csv(OUTPUT_FILE, index=False)

        time.sleep(SLEEP_TIME)

    return df


def main(model):
    df = load_df()
    df = process(df, model)
    df.to_csv(OUTPUT_FILE, index=False)

# model = "openai/gpt-oss-120b"
model = "llama-3.1-8b-instant" #running
main(model)

### **Zero shot prompt**

You are a strict translation system.

Task: Translate the given chunk of sentences into natural, grammatically correct English.

Rules:
- Preserve the original meaning exactly
- Do NOT add or remove information
- Do NOT paraphrase beyond necessary grammar correction
- Fix grammar, spelling, and fluency in English
- Use proper sentence casing and punctuation
- Keep tone neutral and natural

Output:
- Only the final corrected English sentences
- No explanation
- No extra text


### **Few shot Prompt**

You are a strict translation system.

Task: Translate the given chunk of sentences into natural, grammatically correct English.

Rules:

* Preserve the original meaning exactly
* Do NOT add or remove information
* Do NOT paraphrase beyond necessary grammar correction
* Fix grammar, spelling, and fluency in English
* Use proper sentence casing and punctuation
* Keep tone neutral and natural

Output:

* Only the final corrected English sentences
* No explanation
* No extra text

Examples:

1.Input: നന്നായി നിർമ്മിച്ചതും ഓരോ ചില്ലിക്കാശിനും വിലയുള്ളതുമായ മികച്ച ടാബ്‌ലെറ്റ്
Output: Great tablet for the price well built and worth every penny

2.Input: ഇതിന് രക്ഷാകർതൃ നിയന്ത്രണവും കർഫ്യൂവും ഉണ്ടെന്ന് ഞാൻ ഇഷ്ടപ്പെടുന്നു, എനിക്ക് ഇഷ്ടപ്പെടാത്തത് ചാർജർ അതിൽ കുറവാണെന്ന് തോന്നുന്നു അല്ലെങ്കിൽ വേഗത്തിൽ ചാർജ് ചെയ്യുന്നില്ല, ഇത് സ്ലോ ചാർജ് ആണ്, പക്ഷേ അവൾ അത് ഇഷ്ടപ്പെടുന്നു, അത് കൂടുതൽ ചെയ്യുന്നു ഞാന് പ്രതീക്ഷിച്ചത്. ചൈൽഡ് മോഡ് കാരണം അവൾക്ക് ഇന്റർനെറ്റ് ഉപയോഗിക്കാൻ കഴിയാത്ത പുസ്തകങ്ങൾ ഡൗൺലോഡ് ചെയ്യാൻ കഴിയും, ഇത് കുട്ടികൾക്കും മുതിർന്നവർക്കും ഒരു മികച്ച ടാബ്‌ലെറ്റാണ്
Output: I like that it has parental control and a curfew on it, what I don't like is the charger it seemed to have a short in it or doesn't charge fast it's a slow charge but she loves it and it does way more then I expected. She is able to download books she can't use the internet because of the child mode it's a great tablet for children and adults

3.Input: മികച്ച ഓൺലൈൻ അനുഭവം. പ്രായമായവർക്ക് ഇത് എളുപ്പമാക്കിയതിന് നന്ദി, അതിലൂടെ അവർക്ക് എന്തുചെയ്യണമെന്ന് മനസ്സിലാക്കാനാകും. ആകർഷണീയമായ വാങ്ങൽ, വീട്ടിൽ ഷോപ്പുചെയ്യുന്നത് ആകർഷണീയമായിരുന്നു. നന്ദി, ബെസ്റ്റ് ബൈ. രാജ്ഞി
Output: Great online experience. Thanks for making it easy for older people so they can understand what to do. Awesome buy and it was awesome to shop at home. THANKS, Best Buy. Queenie

4.Input: ക്രിസ്മസിന് എന്റെ സുഹൃത്ത് എനിക്ക് ഇത് കൊണ്ടുവന്നു. സംഗീതം സ്ട്രീം ചെയ്യുന്നത് വളരെ നല്ലതാണ്. എന്റെ ഭാര്യക്ക് എനിക്ക് ഒരു പ്രൈം അക്കൗണ്ട് ലഭിച്ചു, ഇപ്പോൾ നമുക്ക് സിനിമകൾ കാണാനും സംഗീതം കേൾക്കാനും അലക്‌സയോട് ആവശ്യപ്പെടാനും കഴിയും.
Output: My buddy got me this for Christmas. It is great to stream music. My wife got me a Prime account and we can now watch movies, listen to music, etc just by asking Alexa.

5.Input: അത് എത്ര ചെറുതാണെന്നും സ്‌ക്രീൻ നിറത്തിലുള്ളതാണെന്നും എനിക്ക് ശരിക്കും ഇഷ്ടമാണ്.
Output: I really like how small it is and the fact that the screen is in color.

6.Input: ബ്ലാക്ക് ഫ്രൈഡേയ്‌ക്ക് ഉൽപ്പന്നം വിൽപ്പനയ്‌ക്കെത്തി. അതിന്റെ ഉദ്ദേശ്യം നിറവേറ്റുന്നു. ഗുണനിലവാരം മികച്ചതാണ്, ഞാൻ ഇത് എന്റെ Chromecast-ൽ ഉപയോഗിക്കുന്നു.
Output: The product was on sale for Black Friday. Serves its purpose. Quality is good and I use it with my Chromecast.

7.Input: ക്രിസ്മസിനായി ഞങ്ങൾ ഇത് ഞങ്ങളുടെ മകന് വാങ്ങി, അവന് 9 വയസ്സ്. അവന് ആവശ്യമായത് ഇതാണ്, ധാരാളം ഗെയിമുകൾ, അവന് വീഡിയോകൾ കാണാനാകും, മുതലായവ, പക്ഷേ എളുപ്പത്തിൽ സജ്ജീകരിച്ച രക്ഷാകർതൃ നിയന്ത്രണങ്ങൾക്ക് നന്ദി, ഇത് ഉപയോഗിച്ച് എനിക്ക് അവനിൽ സുരക്ഷിതത്വം തോന്നുന്നു.
Output: We bought this for our son for Christmas, he's 9. It is exactly what he needed, plenty of games, he can watch videos, etc but I can feel safe with him using it thanks to the easily set parental controls.

8.Input: ഞാൻ ഇത് എന്റെ ഭാര്യക്ക് ഒരു എക്കോ ഡോട്ടിനൊപ്പം ഒരു സമ്മാനമായി വാങ്ങി. അവൾ അത് ഇഷ്ടപ്പെടുന്നു. സജ്ജീകരിക്കാനും ഉപയോഗിക്കാനും എളുപ്പമാണ്.
Output: I bought this as a gift for my wife along with an Echo DOT. She loves it. Easy to setup and use.

9.Input: ക്രിസ്മസിന് കുട്ടികൾക്കായി ഇത് ലഭിച്ചു...ഏറ്റവും മികച്ച സമ്മാനം.
Output: Got this for the kids for Xmas...was the BEST gift.

10.Input: എന്റെ പുത്രന്മാരുടെ പ്രൊഫൈലിനായി ഉപകരണം സജ്ജീകരിക്കുന്നതിന് ഞാൻ പ്രതീക്ഷിച്ചതിലും കുറച്ച് സമയമെടുത്തു, പക്ഷേ എനിക്കത് ഇഷ്ടമാണ്. നാവിഗേറ്റ് ചെയ്യാൻ എളുപ്പമാണ്, കൂടാതെ ഒരു ടൺ പ്രീലോഡ് ചെയ്ത പ്രവർത്തനങ്ങളും അവനുവേണ്ടിയുണ്ട്.
Output: Setting up the device for my sons profile took a litthe more time than I expected, but I love it. It's easy to navigate and has a ton of preloaded activities for him.

Now translate:


# **Model**

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))

## **mbart**

### __ME__

In [ ]:
import os
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    MBartForConditionalGeneration,
    MBart50TokenizerFast,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments
)
from transformers.trainer_utils import get_last_checkpoint


# directories
output_dir = "./mbart_model"
final_output_dir = "./mbart_model_final"

os.makedirs(output_dir, exist_ok=True)
os.makedirs(final_output_dir, exist_ok=True)


# 1 Load YOUR CSV DATA
train_df = pd.read_csv("/content/train.csv")
test_df = pd.read_csv("/content/test.csv")

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)


# 2 Load mBART
model_name = "facebook/mbart-large-50-many-to-many-mmt"

tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
model = MBartForConditionalGeneration.from_pretrained(model_name)


tokenizer.src_lang = "en_XX"  #change
target_lang = "ml_IN"  #change
model.generation_config.forced_bos_token_id = tokenizer.lang_code_to_id["ml_IN"]  #change

max_len = 128


# 3 Preprocess (EN → ML)
def preprocess(example):

    model_inputs = tokenizer(
        example["target"],  #change (English input)
        truncation=True,
        max_length=max_len
    )

    labels = tokenizer(
        text_target=example["source"],  #change (Manglish/Malayalam output)
        truncation=True,
        max_length=max_len
    )["input_ids"]

    labels = [
        token if token != tokenizer.pad_token_id else -100
        for token in labels
    ]

    model_inputs["labels"] = labels

    return model_inputs


train_dataset = train_dataset.map(preprocess)
test_dataset = test_dataset.map(preprocess)

train_dataset = train_dataset.remove_columns(["source", "target"])
test_dataset = test_dataset.remove_columns(["source", "target"])


# 4 Data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100
)


# 5 Training arguments
training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    fp16=True,
    logging_steps=100,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    load_best_model_at_end=True
)


# 6 Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator
)


# 7 Resume training if checkpoint exists
last_checkpoint = get_last_checkpoint(output_dir)

if last_checkpoint:
    print(f"Resuming from {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    trainer.train()


# 8 Save final model
trainer.save_model(final_output_dir)
tokenizer.save_pretrained(final_output_dir)

In [ ]:
!pip install evaluate rouge_score

In [ ]:
import evaluate
import pandas as pd

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")


def generate_translation(sentence):

    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        max_length=max_len
    ).to(model.device)

    generated = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id["ml_IN"],  #change
        max_length=max_len
    )

    return tokenizer.decode(generated[0], skip_special_tokens=True)


predictions = []
references = []
rows = []

for _, row in test_df.iterrows():

    source = row["target"]   # English #change
    target = row["source"]   # Manglish/Malayalam #change

    pred = generate_translation(source)

    predictions.append(pred)
    references.append([target])

    bleu_i = bleu.compute(
        predictions=[pred],
        references=[[target]]
    )["bleu"]

    rouge_i = rouge.compute(
        predictions=[pred],
        references=[target]
    )

    meteor_i = meteor.compute(
        predictions=[pred],
        references=[target]
    )["meteor"]

    rows.append({
        "source": source,
        "target": target,
        "prediction": pred,
        "bleu": bleu_i,
        "meteor": meteor_i,
        "rouge1": rouge_i["rouge1"],
        "rouge2": rouge_i["rouge2"],
        "rougeL": rouge_i["rougeL"]
    })


# save CSV
df_scores = pd.DataFrame(rows)
df_scores.to_csv("sentence_level_scores.csv", index=False)


# corpus scores
bleu_score = bleu.compute(
    predictions=predictions,
    references=[[ref] for ref in references]
)

rouge_score = rouge.compute(
    predictions=predictions,
    references=references
)

meteor_score = meteor.compute(
    predictions=predictions,
    references=references
)

print("BLEU:", bleu_score["bleu"])
print("METEOR:", meteor_score["meteor"])

print("ROUGE-1:", rouge_score["rouge1"])
print("ROUGE-2:", rouge_score["rouge2"])
print("ROUGE-L:", rouge_score["rougeL"])


# sample outputs
for i in range(5):
    print("Source:", test_df.iloc[i]["source"])
    print("Target:", test_df.iloc[i]["target"])
    print("Prediction:", predictions[i])
    print()


### **bidirectional**

In [ ]:
!pip install evaluate rouge_score

In [ ]:
import os
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    MBartForConditionalGeneration,
    MBart50TokenizerFast,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments
)

output_dir = "./mbart_bidirectional_model"
final_output_dir = "./mbart_bidirectional_final"

os.makedirs(output_dir, exist_ok=True)
os.makedirs(final_output_dir, exist_ok=True)

train_df = pd.read_csv("Amazon_CM_Bidirectional_9K.csv")
train_dataset = Dataset.from_pandas(train_df)

model_name = "facebook/mbart-large-50-many-to-many-mmt"

tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
model = MBartForConditionalGeneration.from_pretrained(model_name)

max_len = 128

def preprocess(example):
    text = example["source"]
    target = example["target"]

    if text.startswith("<M2E>"):
        tokenizer.src_lang = "ml_IN"
        text = text.replace("<M2E> ", "")
    else:
        tokenizer.src_lang = "en_XX"
        text = text.replace("<E2M> ", "")

    model_inputs = tokenizer(
        text,
        truncation=True,
        max_length=max_len
    )

    labels = tokenizer(
        text_target=target,
        truncation=True,
        max_length=max_len
    )["input_ids"]

    labels = [
        token if token != tokenizer.pad_token_id else -100
        for token in labels
    ]

    model_inputs["labels"] = labels

    return model_inputs

train_dataset = train_dataset.map(preprocess)
train_dataset = train_dataset.remove_columns(
    ["source", "target"]
)
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100
)

training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    fp16=True,
    logging_steps=100,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator
)

trainer.train()

trainer.save_model(final_output_dir)
tokenizer.save_pretrained(final_output_dir)

In [ ]:
import evaluate
import pandas as pd
import torch

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")

model.eval()

test_df = pd.read_csv("test.csv")
test_dataset = Dataset.from_pandas(test_df)

def generate_translation(sentence):
    tokenizer.src_lang = "ml_IN"
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        max_length=max_len
    ).to(model.device)

    generated = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id["en_XX"],
        max_length=max_len
    )

    return tokenizer.decode(generated[0], skip_special_tokens=True)


predictions = []
references = []
rows = []

for _, row in test_df.iterrows():

    source = row["source"]
    target = row["target"]

    pred = generate_translation(source)

    predictions.append(pred)
    references.append([target])

    bleu_i = bleu.compute(
        predictions=[pred],
        references=[[target]]
    )["bleu"]

    rouge_i = rouge.compute(
        predictions=[pred],
        references=[target]
    )

    meteor_i = meteor.compute(
        predictions=[pred],
        references=[target]
    )["meteor"]

    rows.append({
        "source": source,
        "target": target,
        "prediction": pred,
        "bleu": bleu_i,
        "meteor": meteor_i,
        "rouge1": rouge_i["rouge1"],
        "rouge2": rouge_i["rouge2"],
        "rougeL": rouge_i["rougeL"]
    })

df_scores = pd.DataFrame(rows)
df_scores.to_csv("sentence_level_scores.csv", index=False)

bleu_score = bleu.compute(
    predictions=predictions,
    references=[[ref] for ref in references]
)

rouge_score = rouge.compute(
    predictions=predictions,
    references=references
)

meteor_score = meteor.compute(
    predictions=predictions,
    references=references
)

print("BLEU:", bleu_score["bleu"])
print("METEOR:", meteor_score["meteor"])
print("ROUGE-1:", rouge_score["rouge1"])
print("ROUGE-2:", rouge_score["rouge2"])
print("ROUGE-L:", rouge_score["rougeL"])

for i in range(5):
    print("Source:", test_df.iloc[i]["source"])
    print("Target:", test_df.iloc[i]["target"])
    print("Prediction:", predictions[i])
    print()

## **mt5**

In [ ]:
!pip install transformers datasets sentencepiece evaluate sacrebleu rouge_score

### **ME**

In [ ]:
import os
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments
)
from transformers.trainer_utils import get_last_checkpoint

output_dir = "./mt5_model"
final_output_dir = "./mt5_model_final"

os.makedirs(output_dir, exist_ok=True)
os.makedirs(final_output_dir, exist_ok=True)

# ---------------- LOAD ----------------
train_df = pd.read_csv("/content/train.csv")
test_df = pd.read_csv("/content/test.csv")

train_df = train_df.drop(columns=["Unnamed: 0"])
test_df = test_df.drop(columns=["Unnamed: 0"])

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

model_name = "google/mt5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

max_len = 128

# ME: translate Malayalam sentence to correct English meaning
# EM: translate English sentence to correct Malayalam meaning

# ---------------- PREPROCESS ----------------
def preprocess(example):
    inputs = [
        "translate English sentence to correct Malayalam meaning: " + str(x)
        for x in example["target"]
    ]

    targets = [str(x) for x in example["source"]]

    model_inputs = tokenizer(
        inputs,
        truncation=True,
        max_length=max_len,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=targets,
        truncation=True,
        max_length=max_len,
        padding="max_length"
    )["input_ids"]

    labels = [
        [token if token != tokenizer.pad_token_id else -100 for token in label]
        for label in labels
    ]

    model_inputs["labels"] = labels
    return model_inputs

train_dataset = train_dataset.map(preprocess, batched=True)
test_dataset = test_dataset.map(preprocess, batched=True)

train_dataset = train_dataset.remove_columns(["source", "target"])
test_dataset = test_dataset.remove_columns(["source", "target"])

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100
)

# ---------------- TRAINING ----------------
training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",   
    learning_rate=1e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    fp16=False,
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator
)

last_checkpoint = get_last_checkpoint(output_dir)

if last_checkpoint:
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    trainer.train()

trainer.save_model(final_output_dir)
tokenizer.save_pretrained(final_output_dir)

In [ ]:
import evaluate

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")

def generate_translation(sentence):
    input_text = "translate English sentence to correct Malayalam meaning: " + sentence
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=max_len).to(model.device)
    generated = model.generate(**inputs, max_length=max_len)
    return tokenizer.decode(generated[0], skip_special_tokens=True)

output_file = "mt5_sentence_scores.csv"

if os.path.exists(output_file):
    os.remove(output_file)

predictions = []
references = []

for i, row in test_df.iterrows():
    source = row["target"]
    target = row["source"]

    pred = generate_translation(source)
    print(pred)
    predictions.append(pred)
    references.append([target])

    bleu_i = bleu.compute(predictions=[pred], references=[[target]])["bleu"]
    rouge_i = rouge.compute(predictions=[pred], references=[target])
    meteor_i = meteor.compute(predictions=[pred], references=[target])["meteor"]

    row_data = pd.DataFrame([{
        "source": source,
        "target": target,
        "prediction": pred,
        "bleu": bleu_i,
        "meteor": meteor_i,
        "rouge1": rouge_i["rouge1"],
        "rouge2": rouge_i["rouge2"],
        "rougeL": rouge_i["rougeL"]
    }])

    if not os.path.exists(output_file):
        row_data.to_csv(output_file, index=False)
    else:
        row_data.to_csv(output_file, mode='a', header=False, index=False)

bleu_score = bleu.compute(predictions=predictions, references=[[ref] for ref in references])
rouge_score = rouge.compute(predictions=predictions, references=references)
meteor_score = meteor.compute(predictions=predictions, references=references)

print(bleu_score["bleu"])
print(meteor_score["meteor"])
print(rouge_score["rouge1"], rouge_score["rouge2"], rouge_score["rougeL"])

### **Bidirectional**

In [ ]:
import os
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments
)
from transformers.trainer_utils import get_last_checkpoint

output_dir = "./mt5_model"
final_output_dir = "./mt5_model_final"

os.makedirs(output_dir, exist_ok=True)
os.makedirs(final_output_dir, exist_ok=True)

train_df = pd.read_csv("Amazon_CM_Bidirectional_9K.csv")
test_df = pd.read_csv("test.csv")

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

model_name = "google/mt5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

max_len = 128

# ME: translate Malayalam sentence to correct English meaning
# EM: translate English sentence to correct Malayalam meaning

def preprocess(example):
    inputs = []
    targets = []

    for inp, tgt in zip(example["source"], example["target"]):
        if inp.startswith("<M2E>"):
            text = inp.replace("<M2E> ", "")
            inputs.append("translate Malayalam sentence to correct English meaning: " + text)
            targets.append(tgt)
        else:
            text = inp.replace("<E2M> ", "")
            inputs.append("translate English sentence to correct Malayalam meaning: " + text)
            targets.append(tgt)

    model_inputs = tokenizer(
        inputs,
        truncation=True,
        max_length=max_len,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=targets,
        truncation=True,
        max_length=max_len,
        padding="max_length"
    )["input_ids"]

    labels = [
        [token if token != tokenizer.pad_token_id else -100 for token in label]
        for label in labels
    ]

    model_inputs["labels"] = labels
    return model_inputs


def preprocess_eval(example):
    model_inputs = tokenizer(
        example["source"],
        truncation=True,
        max_length=max_len,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=example["target"],
        truncation=True,
        max_length=max_len,
        padding="max_length"
    )["input_ids"]

    labels = [
        [token if token != tokenizer.pad_token_id else -100 for token in label]
        for label in labels
    ]

    model_inputs["labels"] = labels
    return model_inputs


train_dataset = train_dataset.map(preprocess, batched=True)
test_dataset = test_dataset.map(preprocess_eval, batched=True)

train_dataset = train_dataset.remove_columns(["source", "target"])
test_dataset = test_dataset.remove_columns(["source", "target"])

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100
)

training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    fp16=False,
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator
)

last_checkpoint = get_last_checkpoint(output_dir)

if last_checkpoint:
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    trainer.train()

trainer.save_model(final_output_dir)
tokenizer.save_pretrained(final_output_dir)

In [ ]:
import evaluate

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")

def generate_translation(sentence):
    input_text = "translate Malayalam sentence to correct English meaning: " + sentence
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=max_len).to(model.device)
    generated = model.generate(**inputs, max_length=max_len)
    return tokenizer.decode(generated[0], skip_special_tokens=True)

output_file = "mt5_sentence_scores.csv"

if os.path.exists(output_file):
    os.remove(output_file)

predictions = []
references = []

for i, row in test_df.iterrows():
    source = row["source"]
    target = row["target"]

    pred = generate_translation(source)
    print(pred)
    predictions.append(pred)
    references.append([target])

    bleu_i = bleu.compute(predictions=[pred], references=[[target]])["bleu"]
    rouge_i = rouge.compute(predictions=[pred], references=[target])
    meteor_i = meteor.compute(predictions=[pred], references=[target])["meteor"]

    row_data = pd.DataFrame([{
        "source": source,
        "target": target,
        "prediction": pred,
        "bleu": bleu_i,
        "meteor": meteor_i,
        "rouge1": rouge_i["rouge1"],
        "rouge2": rouge_i["rouge2"],
        "rougeL": rouge_i["rougeL"]
    }])

    if not os.path.exists(output_file):
        row_data.to_csv(output_file, index=False)
    else:
        row_data.to_csv(output_file, mode='a', header=False, index=False)

bleu_score = bleu.compute(predictions=predictions, references=[[ref] for ref in references])
rouge_score = rouge.compute(predictions=predictions, references=references)
meteor_score = meteor.compute(predictions=predictions, references=references)

print(bleu_score["bleu"])
print(meteor_score["meteor"])
print(rouge_score["rouge1"], rouge_score["rouge2"], rouge_score["rougeL"])

## **IndicTrans2**

In [ ]:
import os
os.environ["HF_TOKEN"] = "hf_token"

### **Not fine tune**

In [ ]:
!pip uninstall -y transformers
!pip install -U "transformers==4.40.2" accelerate sentencepiece onnx onnxruntime
!pip install -q IndicTransToolkit

In [ ]:
import os
import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

model_name = "ai4bharat/indictrans2-indic-en-1B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 5

src_lang = "mal_Mlym"
tgt_lang = "eng_Latn"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    attn_implementation="flash_attention_2"
).to(DEVICE)

ip = IndicProcessor(inference=True)


def run_translation(INPUT_PATH, OUTPUT_PATH):

    df = pd.read_csv(INPUT_PATH)

    if os.path.exists(OUTPUT_PATH):
        print(f"Resuming {OUTPUT_PATH}...")
        df_out = pd.read_csv(OUTPUT_PATH)
    else:
        print(f"Starting fresh for {INPUT_PATH}...")
        df_out = df.copy()
        df_out["translated_text"] = None

    total_rows = len(df_out)

    start_index = df_out["translated_text"].last_valid_index()
    start_index = 0 if start_index is None else start_index + 1

    print(f"Starting from row: {start_index}")

    for i in range(start_index, total_rows, BATCH_SIZE):

        try:
            print(f"\nProcessing rows {i} to {min(i+BATCH_SIZE, total_rows)}")

            batch_sentences = (
                df_out.iloc[i:i+BATCH_SIZE, 1]
                .astype(str)
                .tolist()
            )

            batch = ip.preprocess_batch(
                batch_sentences,
                src_lang=src_lang,
                tgt_lang=tgt_lang,
            )

            inputs = tokenizer(
                batch,
                truncation=True,
                padding="longest",
                return_tensors="pt",
                return_attention_mask=True,
            ).to(DEVICE)

            with torch.no_grad():
                generated_tokens = model.generate(
                    **inputs,
                    max_length=256,
                    num_beams=5,
                )

            decoded = tokenizer.batch_decode(
                generated_tokens,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True,
            )

            translations = ip.postprocess_batch(decoded, lang=tgt_lang)

            df_out.loc[i:i+len(translations)-1, "translated_text"] = translations

            df_out.to_csv(OUTPUT_PATH, index=False)

            print("Batch saved")

        except Exception as e:
            print(f"Error at row {i}: {e}")
            break

    print(f"\nDone: {OUTPUT_PATH}")

In [ ]:
run_translation("/content/test.csv", "/content/test_translated.csv")

### **fine tune**

In [ ]:
!pip uninstall -y transformers accelerate peft trl
!pip install transformers==4.38.2 accelerate==0.27.2
!pip install -q IndicTransToolkit

In [ ]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments
)
from IndicTransToolkit.processor import IndicProcessor


# ---------------- CONFIG ----------------
model_name = "ai4bharat/indictrans2-indic-en-dist-200M"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

train_path = "/content/train.csv"
test_path = "/content/test.csv"

src_lang = "mal_Mlym"
tgt_lang = "eng_Latn"

max_len = 128

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# clean
train_df = train_df.dropna()
test_df = test_df.dropna()

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    trust_remote_code=True
).to(DEVICE)

ip = IndicProcessor(inference=False)

def preprocess(example):
    src = str(example["source"]).strip()
    tgt = str(example["target"]).strip()

    # preprocess → returns list
    processed = ip.preprocess_batch(
        [src],
        src_lang=src_lang,
        tgt_lang=tgt_lang
    )

    processed = processed[0]

    model_inputs = tokenizer(
        processed,
        truncation=True,
        max_length=max_len
    )

    labels = tokenizer(
        text_target=tgt,
        truncation=True,
        max_length=max_len
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

train_dataset = train_dataset.map(preprocess)
test_dataset = test_dataset.map(preprocess)

train_dataset = train_dataset.remove_columns(["source", "target"])
test_dataset = test_dataset.remove_columns(["source", "target"])

# ---------------- COLLATOR ----------------
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100
)


# ---------------- TRAINING ----------------
training_args = TrainingArguments(
    output_dir="./indictrans_finetuned",
    evaluation_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    logging_steps=100,
    save_strategy="epoch",
    fp16=True,
    report_to="none"
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
)

trainer.train()

trainer.save_model("./indictrans_finetuned")
tokenizer.save_pretrained("./indictrans_finetuned")

In [ ]:
!pip install evaluate rouge_score

In [ ]:
import evaluate
import pandas as pd
import torch

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")

def generate_translation(sentence):

    processed = ip.preprocess_batch(
        [sentence],
        src_lang=src_lang,
        tgt_lang=tgt_lang
    )[0]

    inputs = tokenizer(
        processed,
        return_tensors="pt",
        truncation=True,
        max_length=max_len
    ).to(model.device)

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_length=max_len
        )

    decoded = tokenizer.decode(
        generated[0],
        skip_special_tokens=True
    )

    output = ip.postprocess_batch(
        [decoded],
        lang=tgt_lang
    )[0]

    return output

import evaluate
import pandas as pd
import torch

# reload test_df (FIX)
test_df = pd.read_csv("test.csv")
test_df = test_df.dropna()

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")

predictions = []
references = []
rows = []

for _, row in test_df.iterrows():
    print(source)
    source = str(row["source"]).strip()
    target = str(row["target"]).strip()

    pred = generate_translation(source)

    predictions.append(pred)
    references.append(target)

    bleu_i = bleu.compute(
        predictions=[pred],
        references=[[target]]
    )["bleu"]

    rouge_i = rouge.compute(
        predictions=[pred],
        references=[target]
    )

    meteor_i = meteor.compute(
        predictions=[pred],
        references=[target]
    )["meteor"]
    rows.append({
        "source": source,
        "target": target,
        "prediction": pred,
        "bleu": bleu_i,
        "meteor": meteor_i,
        "rouge1": rouge_i["rouge1"],
        "rouge2": rouge_i["rouge2"],
        "rougeL": rouge_i["rougeL"]
    })


df_scores = pd.DataFrame(rows)
df_scores.to_csv("sentence_level_scores.csv", index=False)


bleu_score = bleu.compute(
    predictions=predictions,
    references=[[ref] for ref in references]
)

rouge_score = rouge.compute(
    predictions=predictions,
    references=references
)

meteor_score = meteor.compute(
    predictions=predictions,
    references=references
)

print("===== CORPUS SCORES =====")
print("BLEU:", bleu_score["bleu"])
print("METEOR:", meteor_score["meteor"])
print("ROUGE-1:", rouge_score["rouge1"])
print("ROUGE-2:", rouge_score["rouge2"])
print("ROUGE-L:", rouge_score["rougeL"])


# ---------------- SAMPLE OUTPUT ----------------
for i in range(5):
    print("Source:", test_df.iloc[i]["source"])
    print("Target:", test_df.iloc[i]["target"])
    print("Prediction:", predictions[i])
    print()

## **T5**

In [ ]:
!pip install transformers datasets sentencepiece evaluate sacrebleu rouge_score

### **ME**

In [ ]:
import os
import pandas as pd
import torch
from datasets import Dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, DataCollatorForSeq2Seq, Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

output_dir = "./t5_model"
final_output_dir = "./t5_model_final"

os.makedirs(output_dir, exist_ok=True)
os.makedirs(final_output_dir, exist_ok=True)

train_df = pd.read_csv("/content/train.csv")
test_df = pd.read_csv("/content/test.csv")

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

model_name = "t5-base"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

max_len = 128

def preprocess(example):
    inputs = ["translate English sentence to correct Malayalam meaning: " + x for x in example["target"]]
    model_inputs = tokenizer(inputs, truncation=True, max_length=max_len)
    labels = tokenizer(text_target=example["source"], truncation=True, max_length=max_len)["input_ids"]
    labels = [[token if token != tokenizer.pad_token_id else -100 for token in label] for label in labels]
    model_inputs["labels"] = labels
    return model_inputs

train_dataset = train_dataset.map(preprocess, batched=True)
test_dataset = test_dataset.map(preprocess, batched=True)

train_dataset = train_dataset.remove_columns(["source", "target"])
test_dataset = test_dataset.remove_columns(["source", "target"])

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)

training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    fp16=True,
    logging_steps=100,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator
)

last_checkpoint = get_last_checkpoint(output_dir)

if last_checkpoint:
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    trainer.train()

trainer.save_model(final_output_dir)
tokenizer.save_pretrained(final_output_dir)

In [ ]:
import evaluate

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")

def generate_translation(sentence):
    input_text = "translate English sentence to correct Malayalam meaning: " + sentence
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=max_len).to(model.device)
    generated = model.generate(**inputs, max_length=max_len)
    return tokenizer.decode(generated[0], skip_special_tokens=True)

output_file = "t5_sentence_scores.csv"

if os.path.exists(output_file):
    os.remove(output_file)

predictions = []
references = []

for i, row in test_df.iterrows():
    source = str(row["target"])
    target = str(row["source"])

    pred = generate_translation(source)
    print("source",source)
    print("target",target)
    print("pred", pred)
    if not target or not pred:
        continue
    predictions.append(pred)
    references.append([target])

    bleu_i = bleu.compute(
        predictions=[pred],
        references=[[target]]
    )["bleu"]

    rouge_i = rouge.compute(
        predictions=[pred],
        references=[target]
    )

    meteor_i = meteor.compute(
        predictions=[pred],
        references=[target]
    )["meteor"]

    row_data = pd.DataFrame([{
        "source": source,
        "target": target,
        "prediction": pred,
        "bleu": bleu_i,
        "meteor": meteor_i,
        "rouge1": rouge_i["rouge1"],
        "rouge2": rouge_i["rouge2"],
        "rougeL": rouge_i["rougeL"]
    }])

    if not os.path.exists(output_file):
        row_data.to_csv(output_file, index=False)
    else:
        row_data.to_csv(output_file, mode='a', header=False, index=False)

bleu_score = bleu.compute(predictions=predictions, references=[[ref] for ref in references])
rouge_score = rouge.compute(predictions=predictions, references=references)
meteor_score = meteor.compute(predictions=predictions, references=references)

print(bleu_score["bleu"])
print(meteor_score["meteor"])
print(rouge_score["rouge1"], rouge_score["rouge2"], rouge_score["rougeL"])

### __Bidirectional__

In [ ]:
!pip install transformers datasets sentencepiece evaluate sacrebleu rouge_score

In [ ]:
import os
import pandas as pd
import torch
from datasets import Dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, DataCollatorForSeq2Seq, Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

output_dir = "./t5_model"
final_output_dir = "./t5_model_final"

os.makedirs(output_dir, exist_ok=True)
os.makedirs(final_output_dir, exist_ok=True)

train_df = pd.read_csv("/content/Amazon_CM_Bidirectional_9K.csv")
test_df = pd.read_csv("/content/test.csv")

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

model_name = "t5-base"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

max_len = 128


def preprocess(example):
    inputs = []
    targets = []

    for inp, tgt in zip(example["source"], example["target"]):
        if inp.startswith("<M2E>"):
            text = inp.replace("<M2E> ", "")
            inputs.append("translate Malayalam sentence to correct English meaning: " + text)
            targets.append(tgt)
        else:
            text = inp.replace("<E2M> ", "")
            inputs.append("translate English sentence to correct Malayalam meaning: " + text)
            targets.append(tgt)

    model_inputs = tokenizer(
        inputs,
        truncation=True,
        max_length=max_len,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=targets,
        truncation=True,
        max_length=max_len,
        padding="max_length"
    )["input_ids"]

    labels = [
        [token if token != tokenizer.pad_token_id else -100 for token in label]
        for label in labels
    ]

    model_inputs["labels"] = labels
    return model_inputs


def preprocess_eval(example):
    model_inputs = tokenizer(
        example["source"],
        truncation=True,
        max_length=max_len,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=example["target"],
        truncation=True,
        max_length=max_len,
        padding="max_length"
    )["input_ids"]

    labels = [
        [token if token != tokenizer.pad_token_id else -100 for token in label]
        for label in labels
    ]

    model_inputs["labels"] = labels
    return model_inputs


train_dataset = train_dataset.map(preprocess, batched=True)
test_dataset = test_dataset.map(preprocess_eval, batched=True)

train_dataset = train_dataset.remove_columns(["source", "target"])
test_dataset = test_dataset.remove_columns(["source", "target"])

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)

training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    fp16=True,
    logging_steps=100,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator
)

last_checkpoint = get_last_checkpoint(output_dir)

if last_checkpoint:
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    trainer.train()

trainer.save_model(final_output_dir)
tokenizer.save_pretrained(final_output_dir)

In [ ]:
import os
import evaluate
import pandas as pd
import torch

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")

model.eval()

def generate_translation(sentence):
    input_text = "translate Malayalam sentence to correct English meaning: " + sentence
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_len
    ).to(model.device)

    generated = model.generate(
        **inputs,
        max_length=max_len
    )

    return tokenizer.decode(generated[0], skip_special_tokens=True)


output_file = "t5_sentence_scores.csv"

if os.path.exists(output_file):
    os.remove(output_file)

predictions = []
references = []

for i, row in test_df.iterrows():
    source = str(row["source"])
    target = str(row["target"])

    pred = generate_translation(source)

    if not target or not pred:
        continue

    predictions.append(pred)
    references.append([target])

    bleu_i = bleu.compute(
        predictions=[pred],
        references=[target]
    )["bleu"]

    rouge_i = rouge.compute(
        predictions=[pred],
        references=[target]
    )

    meteor_i = meteor.compute(
        predictions=[pred],
        references=[target]
    )["meteor"]

    row_data = pd.DataFrame([{
        "source": source,
        "target": target,
        "prediction": pred,
        "bleu": bleu_i,
        "meteor": meteor_i,
        "rouge1": rouge_i["rouge1"],
        "rouge2": rouge_i["rouge2"],
        "rougeL": rouge_i["rougeL"]
    }])

    if not os.path.exists(output_file):
        row_data.to_csv(output_file, index=False)
    else:
        row_data.to_csv(output_file, mode='a', header=False, index=False)

bleu_score = bleu.compute(
    predictions=predictions,
    references=references
)

rouge_score = rouge.compute(
    predictions=predictions,
    references=[ref[0] for ref in references]
)

meteor_score = meteor.compute(
    predictions=predictions,
    references=[ref[0] for ref in references]
)

print(bleu_score["bleu"])
print(meteor_score["meteor"])
print(rouge_score["rouge1"], rouge_score["rouge2"], rouge_score["rougeL"])

## **Evaluation**

In [ ]:
!pip install evaluate rouge_score

In [ ]:
import pandas as pd
import evaluate


def load_metrics():
    return {
        "bleu": evaluate.load("bleu"),
        "rouge": evaluate.load("rouge"),
        "meteor": evaluate.load("meteor")
    }


def prepare_data(df, pred_col, ref_col, source_col):
    # Keep all required columns + remove NaNs
    print("before:", len(df))
    df = df[[source_col, pred_col, ref_col]].dropna().reset_index(drop=True)
    print("after :", len(df))
    predictions = df[pred_col].astype(str).str.strip().tolist()
    references = [[ref] for ref in df[ref_col].astype(str).str.strip().tolist()]

    return df, predictions, references


def compute_sentence_scores(df, predictions, references, metrics, source_col, pred_col, ref_col):
    results = []

    for i, (pred, ref) in enumerate(zip(predictions, references)):
        ref_text = ref[0]
        source_text = df.iloc[i][source_col]

        bleu_score = metrics["bleu"].compute(
            predictions=[pred],
            references=[[ref_text]]
        )["bleu"]

        rouge_score = metrics["rouge"].compute(
            predictions=[pred],
            references=[ref_text]
        )

        meteor_score = metrics["meteor"].compute(
            predictions=[pred],
            references=[ref_text]
        )["meteor"]

        results.append({
            "source": source_text,
            "prediction": pred,
            "reference": ref_text,
            "BLEU": bleu_score,
            "METEOR": meteor_score,
            "ROUGE-1": rouge_score["rouge1"],
            "ROUGE-2": rouge_score["rouge2"],
            "ROUGE-L": rouge_score["rougeL"]
        })

    return pd.DataFrame(results)


# ---------------- PRINT SAMPLES ----------------
def print_samples(df, n=5):
    print("\n===== SAMPLE OUTPUTS =====")
    for i in range(min(n, len(df))):
        print(f"\n--- Example {i} ---")
        print("Source:", df.iloc[i]["source"])
        print("Target:", df.iloc[i]["reference"])
        print("Prediction:", df.iloc[i]["prediction"])


# ---------------- MAIN FUNCTION ----------------
def evaluate_csv(
    file_path,
    pred_col,
    ref_col,
    source_col="source",
    show_samples=True,
    n_samples=5
):
    df = pd.read_csv(file_path)

    metrics = load_metrics()
    clean_df, predictions, references = prepare_data(df, pred_col, ref_col, source_col)

    scores_df = compute_sentence_scores(
        clean_df,
        predictions,
        references,
        metrics,
        source_col,
        pred_col,
        ref_col
    )

    # ---- Average Scores ----
    print("\n===== AVERAGE SCORES =====")
    print(scores_df[["BLEU", "METEOR", "ROUGE-1", "ROUGE-2", "ROUGE-L"]].mean())

    # ---- Samples ----
    if show_samples:
        print_samples(scores_df, n_samples)

    return scores_df


# ---------------- USAGE ----------------
if __name__ == "__main__":
    results_df = evaluate_csv(
        file_path="Llama_fewshot.csv",
        pred_col="llama_fewshot",
        ref_col="target",
        source_col="source"
    )

    results_df.to_csv("Llama_fewshot_eval.csv", index=False)

## **Analysis**